In [ ]:
# Store Sales Prediction
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

: 

In [ ]:
train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
oil = pd.read_csv("oil.csv")
holidays = pd.read_csv("holidays_events.csv")

In [ ]:
train.head()
train.info()
train.describe()
train.shape
train.columns

In [ ]:
train.isnull().sum()

In [ ]:
train.duplicated().sum()

In [ ]:
train.drop_duplicates(inplace=True)

In [ ]:
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

In [ ]:
df = pd.merge(
    train,
    stores,
    on='store_nbr',
    how='left'
)

In [ ]:
df = pd.merge(
    df,
    oil,
    on='date',
    how='left'
)

In [ ]:
df = pd.merge(
    df,
    holidays,
    on='date',
    how='left'
)

In [ ]:
df.head()
df.info()
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill')
df['dcoilwtico'] = df['dcoilwtico'].fillna(method='bfill')

In [ ]:
df.fillna("Unknown", inplace=True)

In [ ]:
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month
df['Day'] = df['date'].dt.day
df['DayOfWeek'] = df['date'].dt.dayofweek
df['WeekOfYear'] = df['date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['date'].dt.quarter

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
encoder = LabelEncoder()

categorical_columns = df.select_dtypes(include='object').columns
for column in categorical_columns:
    df[column] = df[column].astype(str)
    df[column] = encoder.fit_transform(df[column])

In [ ]:
df.head()
df.info()
df.describe()

In [ ]:
X = df.drop(['sales', 'date'], axis=1)
y = df['sales']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
print("Training Data :", X_train.shape)
print("Testing Data :", X_test.shape)

In [ ]:
plt.style.use('ggplot')
sns.set(font_scale=1.1)
# Increase default figure size
plt.rcParams['figure.figsize'] = (10,6)

In [ ]:
print("Shape of Dataset:", df.shape)
print("\nColumn Names:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)

In [ ]:
df.describe().T

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0]
print(missing)

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
plt.figure(figsize=(10,5))
sns.barplot(
    x=missing.index,
    y=missing.values
)
plt.xticks(rotation=90)
plt.title("Missing Values")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df["sales"],
    bins=60,
    kde=True
)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x=df["sales"]
)
plt.title("Boxplot of Sales")
plt.show()

In [ ]:
monthly_sales = df.groupby("Month")["sales"].mean()
plt.figure(figsize=(10,5))
monthly_sales.plot(
    marker='o'
)
plt.title("Average Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.grid(True)
plt.show()

In [ ]:
year_sales = df.groupby("Year")["sales"].mean()

plt.figure(figsize=(8,5))
year_sales.plot(
    kind='bar'
)
plt.title("Average Sales by Year")
plt.ylabel("Sales")
plt.show()

In [ ]:
family_sales = df.groupby("family")["sales"].sum()

family_sales = family_sales.sort_values(ascending=False).head(15)
plt.figure(figsize=(12,6))
sns.barplot(
    x=family_sales.index,
    y=family_sales.values
)
plt.xticks(rotation=90)
plt.title("Top 15 Product Families")
plt.show()

In [ ]:
store_sales = df.groupby("store_nbr")["sales"].mean()

plt.figure(figsize=(14,5))
store_sales.plot()
plt.title("Average Sales per Store")
plt.xlabel("Store Number")
plt.ylabel("Average Sales")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x="onpromotion",
    y="sales",
    alpha=0.5
)
plt.title("Promotion vs Sales")
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    df["dcoilwtico"]
)
plt.title("Oil Price Trend")
plt.xlabel("Records")
plt.ylabel("Oil Price")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    x="type_y",
    data=df
)
plt.title("Holiday Types")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12,8))

corr = df.corr(numeric_only=True)
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
corr["sales"].sort_values(ascending=False)

In [ ]:
selected = df[
    [
        "sales",
        "onpromotion",
        "dcoilwtico",
        "Month",
        "DayOfWeek"
    ]
]
sns.pairplot(selected)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=df,
    x="DayOfWeek",
    y="sales"
)
plt.title("Sales by Day of Week")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=df,
    x="cluster",
    y="sales"
)
plt.title("Sales by Store Cluster")
plt.show()

In [ ]:
numeric_columns = [
    "sales",
    "onpromotion",
    "dcoilwtico"
]

for col in numeric_columns:

    plt.figure(figsize=(8,4))

    sns.boxplot(
        x=df[col]
    )

    plt.title(col)
    plt.show()

In [ ]:
# Linear Regression
lr = LinearRegression()
# Decision Tree
dt = DecisionTreeRegressor(
    random_state=42
)
# Random Forest
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
print("Training Linear Regression...")
lr.fit(X_train, y_train)
print("Training Completed")

In [ ]:
lr_pred = lr.predict(X_test)

In [ ]:
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_mse = mean_squared_error(y_test, lr_pred)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_pred)
print("Linear Regression")
print("MAE :", lr_mae)
print("MSE :", lr_mse)
print("RMSE :", lr_rmse)
print("R2 Score :", lr_r2)

In [ ]:
print("Training Decision Tree...")
dt.fit(X_train, y_train)
print("Training Completed")

In [ ]:
dt_pred = dt.predict(X_test)

In [ ]:
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_test, dt_pred)
print("Decision Tree")
print("MAE :", dt_mae)
print("MSE :", dt_mse)
print("RMSE :", dt_rmse)
print("R2 Score :", dt_r2)

In [ ]:
results = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "Decision Tree"
    ],

    "MAE": [
        lr_mae,
        dt_mae
    ],

    "MSE": [
        lr_mse,
        dt_mse
    ],

    "RMSE": [
        lr_rmse,
        dt_rmse
    ],

    "R2 Score": [
        lr_r2,
        dt_r2
    ]

})

print(results)

In [ ]:
best_model = results.loc[results["R2 Score"].idxmax()]

print("Best Model")
print(best_model)

In [ ]:
plt.figure(figsize=(7,5))

sns.barplot(
    data=results,
    x="Model",
    y="R2 Score",
    palette="viridis"
)
plt.title("Model Comparison based on R² Score")
plt.ylabel("R² Score")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.barplot(
    data=results,
    x="Model",
    y="RMSE",
    palette="Set2"
)
plt.title("Model Comparison based on RMSE")
plt.ylabel("RMSE")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.barplot(
    data=results,
    x="Model",
    y="MAE",
    palette="coolwarm"
)
plt.title("Model Comparison based on MAE")
plt.ylabel("MAE")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    dt_pred,
    alpha=0.5
)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales (Decision Tree)")
plt.show()

In [ ]:
residuals = y_test - dt_pred

plt.figure(figsize=(8,5))

sns.histplot(
    residuals,
    bins=40,
    kde=True
)
plt.title("Residual Distribution (Decision Tree)")
plt.xlabel("Residual")
plt.show()

In [ ]:
import joblib

joblib.dump(
    dt,
    "DecisionTreeModel.pkl"
)
print("Decision Tree Model Saved Successfully")

In [ ]:
import joblib

joblib.dump(
    lr,
    "LinearRegressionModel.pkl"
)
print("Linear Regression Model Saved Successfully")

In [ ]:
model = joblib.load("DecisionTreeModel.pkl")

In [ ]:
model = joblib.load("LinearRegressionModel.pkl")

In [ ]:
print("="*50)
print("Linear Regression Performance")
print("="*50)
print(f"MAE      : {lr_mae:.4f}")
print(f"MSE      : {lr_mse:.4f}")
print(f"RMSE     : {lr_rmse:.4f}")
print(f"R² Score : {lr_r2:.4f}")

print("\n" + "="*50)
print("Decision Tree Performance")
print("="*50)
print(f"MAE      : {dt_mae:.4f}")
print(f"MSE      : {dt_mse:.4f}")
print(f"RMSE     : {dt_rmse:.4f}")
print(f"R² Score : {dt_r2:.4f}")